# E6 | Model Forecast Prophet
Treinar Prophet para D+1 e D+7 com MLflow

## 📋 Objetivo

Neste notebook, vou treinar um modelo **Prophet** para prever o volume de incidentes com **7 dias de antecedência** (D+1 a D+7). O Prophet é ideal para séries temporais com sazonalidade forte e regressores externos.

### Por que Prophet?
- **Sazonalidade**: Captura padrões semanais (mais incidentes na segunda-feira?) e anuais
- **Regressores**: Incorpora fatores externos (fim de semana, taxa de violação de SLA)
- **Interpretável**: Decompõe a previsão em componentes (trend, sazonal)

### Fluxo
1. **Setup**: Conexão com RDS
2. **Carregamento**: Dados históricos de volume diário
3. **Preparação**: Criação de regressores (fim de semana, taxa de violação)
4. **Treino/Teste**: Split 80/20 nos últimos 30 dias
5. **Avaliação**: MAPE e MAE no conjunto de teste
6. **Forecast**: Previsões para D+1 a D+7
7. **Logging**: Rastreamento no MLflow

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import mlflow

load_dotenv()
print('✅ Imports OK')

✅ Imports OK


In [ ]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

print(f'Host: {RDS_HOST}')
print(f'DB:   {RDS_DATABASE}')
print(f'User: {RDS_USER}')

connection_url = f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"

%load_ext sql
%sql {connection_url}

Host: terraform-20260518150028461700000001.c4xegmk24lg6.us-east-1.rds.amazonaws.com
DB:   aiops_gold
User: postgres
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 📥 Etapa 1: Carregamento de Dados

Vou carregar os dados de previsão do RDS. Estes dados já foram processados na camada **gold** do data lake e contêm:
- **data_abertura**: Data do incidente (timestamp)
- **total_chamados**: Volume diário de incidentes (target)
- **is_fim_de_semana**: Indicador de fim de semana (regressador)
- **pct_violacao_sla**: Percentual de violação de SLA (regressador)

### 📊 Dados Carregados

Os dados cobrem **~1 ano de histórico** (2025 inteiro), com 121.263 registros diários. Este volume histórico é essencial para que o Prophet aprenda:
- **Trend**: Tendência geral do volume de incidentes
- **Sazonalidade semanal**: Segundas-feiras têm mais incidentes?
- **Sazonalidade anual**: Períodos de maior carga (Black Friday, férias)?

**Próximo passo**: Vou preparar os dados no formato que o Prophet espera (ds, y, regressadores).

## 🔧 Etapa 2: Preparação dos Dados

O Prophet requer um DataFrame com colunas específicas:
- **ds**: Timestamp (data)
- **y**: Target (volume de incidentes)
- **Regressadores**: Variáveis externas que influenciam o target

Vou criar dois regressadores:
1. **fim_de_semana** (0/1): Permite ao modelo aprender que fins de semana têm padrões diferentes
2. **taxa_violacao** (0-1): A taxa de violação de SLA pode estar correlacionada com volume

## ✂️ Etapa 3: Split Train/Test

Vou usar os **últimos 30 dias** como conjunto de teste. Por quê?
- **Avaliação realista**: Simula prever 30 dias no passado (dados que o modelo não viu)
- **Séries temporais**: Não fazemos split aleatório (quebra a ordem temporal)
- **30 dias**: Período suficiente para avaliar sazonalidade semanal (~4 semanas)

## 🚀 Etapa 4: Treino do Modelo Prophet

Vou configurar o Prophet com:
- **yearly_seasonality=True**: Aprende padrões anuais
- **weekly_seasonality=True**: Aprende padrões semanais (segunda-feira vs domingo)
- **seasonality_mode='additive'**: A sazonalidade é SOMADA ao trend (melhor para volumes absolutos)
- **Regressadores**: Fim de semana e taxa de violação são incorporados ao modelo

O Prophet usa **Stan** (biblioteca de probabilidade) para encontrar os melhores parâmetros. Pode levar alguns segundos.

## 📈 Etapa 5: Avaliação no Conjunto de Teste

Vou usar duas métricas:
- **MAPE** (Mean Absolute Percentage Error): Erro percentual médio. Meta: < 20%
  - Útil para volume de incidentes (entendemos como "20% de erro no volume")
- **MAE** (Mean Absolute Error): Erro absoluto médio em chamados
  - Complementa MAPE: se MAPE=10%, quantos chamados isso representa?

Vou fazer previsão nos 30 dias de teste e comparar com o volume real.

## 🔮 Etapa 6: Forecast D+1 a D+7 (Próxima Semana)

Agora vou fazer **previsões reais** para os próximos 7 dias! O modelo:
1. Usa toda a série histórica (train + test) para treinar novamente
2. Projeta para D+1 a D+7
3. Incorpora:
   - **Trend**: Tendência do volume histórico
   - **Sazonalidade**: Padrão do dia da semana (segunda-feira vs domingo)
   - **Regressadores**: Se segunda é fim de semana (não), qual é a taxa de SLA esperada?

**Output**: Volume previsto para cada dia + intervalo de confiança (95%)

## 📦 Etapa 7: Logging no MLflow e Salvamento de Resultados

Vou:
1. **Rastrear no MLflow**: Registrar parâmetros (sazonalidade), métricas (MAPE, MAE) e artefatos (modelo)
   - Útil para comparar com futuras versões do modelo
   - Auditoria: quem treinou, quando, com quais parâmetros
2. **Salvar resultados locais**:
   - **forecast_d1_d7.csv**: Previsões D+1 a D+7 para usar em dashboards
   - **prophet_test_metrics.csv**: Avaliação no teste (erros por dia)
   - **prophet_summary.csv**: Resumo de métricas

In [ ]:
# Carregar dados para XGBoost
from sqlalchemy import create_engine

# Reutilizar credenciais da célula anterior (RDS_HOST, RDS_USER, RDS_PASSWORD, RDS_DATABASE)
engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

In [ ]:
# Ler e agregar dados por dia (CORRIGIDO)
query = '''
    SELECT DATE(data_abertura) AS data_abertura,
           COUNT(*) AS total_chamados,
           MAX(CAST(is_fim_de_semana AS INT)) AS is_fim_de_semana,
           AVG(CAST(pct_violacao_sla AS FLOAT)) AS pct_violacao_sla
    FROM gold_ml.ml_forecast_dataset
    GROUP BY DATE(data_abertura)
    ORDER BY data_abertura
'''
df = pd.read_sql(query, engine)
df['data_abertura'] = pd.to_datetime(df['data_abertura'])
print(f'✅ Loaded {len(df)} DIAS from {df.data_abertura.min().date()} to {df.data_abertura.max().date()}')
print(f'   Volume médio: {df.total_chamados.mean():.0f} chamados/dia')
print(f'   Min/Max: {df.total_chamados.min():.0f} / {df.total_chamados.max():.0f}')

✅ Loaded 365 DIAS from 2025-01-01 to 2025-12-31
   Volume médio: 332 chamados/dia
   Min/Max: 25 / 1419


In [ ]:
# Preparar Prophet dataset
df_p = df[['data_abertura', 'total_chamados']].rename(columns={'data_abertura': 'ds', 'total_chamados': 'y'})
df_p['ds'] = pd.to_datetime(df_p['ds'])
df_p['fim_de_semana'] = df['is_fim_de_semana'].values
df_p['taxa_violacao'] = df['pct_violacao_sla'].fillna(df['pct_violacao_sla'].mean()).values
df_p = df_p.sort_values('ds').reset_index(drop=True)
print(f'Ready: {df_p.shape}')

Ready: (365, 4)


In [ ]:
# Split
train = df_p.iloc[:-30]
test = df_p.iloc[-30:]
print(f'Train: {len(train)}, Test: {len(test)}')

Train: 335, Test: 30


In [ ]:
# Treinar
model = Prophet(yearly_seasonality=True, weekly_seasonality=True, seasonality_mode='additive')
model.add_regressor('fim_de_semana')
model.add_regressor('taxa_violacao')
print('Training...')
model.fit(train)
print('✅ Done')

Training...


11:50:04 - cmdstanpy - INFO - Chain [1] start processing
11:50:04 - cmdstanpy - INFO - Chain [1] done processing


✅ Done


In [ ]:
# Avaliar
forecast = model.predict(test[['ds', 'fim_de_semana', 'taxa_violacao']])
eval = test[['y']].copy()
eval['yhat'] = forecast['yhat'].values

mape = mean_absolute_percentage_error(eval['y'], eval['yhat'])
mae = (eval['y'] - eval['yhat']).abs().mean()
print(f'MAPE: {mape:.2%}, MAE: {mae:.0f}')

MAPE: 16.36%, MAE: 139


In [ ]:
# Forecast D+1 a D+7
future_dates = pd.date_range(start=pd.Timestamp(datetime.now()).normalize() + timedelta(days=1), periods=7)
future = pd.DataFrame({
    'ds': future_dates,
    'fim_de_semana': [1 if d.dayofweek in [5,6] else 0 for d in future_dates],
    'taxa_violacao': df_p['taxa_violacao'].mean()
})

forecast_future = model.predict(future)
for i, row in forecast_future.iterrows():
    yhat = int(row['yhat'])
    date = row['ds'].strftime('%d/%m')
    print(f'D+{i+1} ({date}): {yhat} chamados')

D+1 (21/05): 1178 chamados
D+2 (22/05): 1158 chamados
D+3 (23/05): 1075 chamados
D+4 (24/05): 1036 chamados
D+5 (25/05): 1152 chamados
D+6 (26/05): 1159 chamados
D+7 (27/05): 1173 chamados


In [ ]:
# MLflow: Rastreamento completo do modelo
import joblib
import tempfile
import matplotlib.pyplot as plt
from mlflow.models import infer_signature

mlflow.set_experiment('prophet_forecast')
with mlflow.start_run(run_name='prophet_v1_aggregated_daily'):
    # 1. Log de parâmetros completo
    mlflow.log_params({
        'model_type': 'Prophet',
        'seasonality_mode': 'additive',
        'yearly_seasonality': True,
        'weekly_seasonality': True,
        'train_days': len(train),
        'test_days': len(test),
        'regressors': 'fim_de_semana,taxa_violacao'
    })
    
    # 2. Log de métricas
    mlflow.log_metrics({
        'mape': mape,
        'mae': mae,
        'train_size': len(train),
        'test_size': len(test)
    })
    
    # 3. Tags de contexto
    mlflow.set_tag('model_type', 'Prophet - Forecast')
    mlflow.set_tag('task', 'incidente_volume_forecast_d1_d7')
    mlflow.set_tag('data_version', f"{df_p['ds'].min().date()}_to_{df_p['ds'].max().date()}")
    mlflow.set_tag('notebook', '05_model_forecast_prophet')
    mlflow.set_tag('target_variable', 'total_chamados')
    mlflow.set_tag('evaluation_metric', 'MAPE')
    
    # 4. Log do modelo
    temp_model_path = os.path.join(tempfile.gettempdir(), 'prophet_model.pkl')
    joblib.dump(model, temp_model_path)
    mlflow.log_artifact(temp_model_path, 'model')
    
    # 5. Log de artefatos de avaliação (gráfico de erro)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Gráfico 1: Série temporal com forecast no teste
    ax1.plot(train['ds'], train['y'], label='Train', linewidth=1.5, alpha=0.7)
    ax1.plot(test['ds'], test['y'], label='Test (Real)', linewidth=2, color='blue')
    ax1.plot(test['ds'], eval['yhat'], label='Forecast', linewidth=2, color='red', linestyle='--')
    ax1.fill_between(test['ds'], 
                      forecast[forecast['ds'].isin(test['ds'])]['yhat_lower'].values,
                      forecast[forecast['ds'].isin(test['ds'])]['yhat_upper'].values,
                      alpha=0.2, color='red', label='95% Confidence')
    ax1.set_title('Prophet Forecast vs Actual (Test Set)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Data')
    ax1.set_ylabel('Incidentes')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Gráfico 2: Erros por dia
    errors = (eval['y'] - eval['yhat']).abs()
    ax2.bar(range(len(errors)), errors.values, color='coral', alpha=0.7)
    ax2.axhline(y=errors.mean(), color='red', linestyle='--', label=f'Mean Error: {errors.mean():.0f}')
    ax2.set_title('Absolute Errors (Test Set)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Days')
    ax2.set_ylabel('Absolute Error (chamados)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Salvar figura
    temp_fig_path = os.path.join(tempfile.gettempdir(), 'prophet_evaluation.png')
    plt.savefig(temp_fig_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(temp_fig_path, 'evaluation')
    plt.close()
    
    print('✅ MLflow logging completo:')
    print(f'   - Run name: prophet_v1_aggregated_daily')
    print(f'   - Métricas: MAPE={mape:.2%}, MAE={mae:.0f}')
    print(f'   - Artefatos: modelo + gráfico de avaliação')
    
    run_id = mlflow.active_run().info.run_id
    print(f'\n🏃 View run at: https://mlflow.looplyai.com.br/#/experiments/10/runs/{run_id}')

✅ MLflow logging completo:
   - Run name: prophet_v1_aggregated_daily
   - Métricas: MAPE=16.36%, MAE=139
   - Artefatos: modelo + gráfico de avaliação

🏃 View run at: https://mlflow.looplyai.com.br/#/experiments/10/runs/be5464a8de7943a4a05eeddcc5f3d853
🏃 View run prophet_v1_aggregated_daily at: https://mlflow.looplyai.com.br/#/experiments/10/runs/be5464a8de7943a4a05eeddcc5f3d853
🧪 View experiment at: https://mlflow.looplyai.com.br/#/experiments/10


In [ ]:
# CROSS-VALIDATION — Prophet com janelas móveis (rolling window)
from prophet.diagnostics import cross_validation, performance_metrics

print('🔄 Executando Cross-Validation do Prophet (este processo leva alguns minutos)...')

# Configurar CV com windows móveis
# initial: 90 dias de dados iniciais para o primeiro modelo
# period: a cada 30 dias, treina novo modelo
# horizon: 7 dias de previsão à frente (queremos avaliar D+1 a D+7)
df_cv = cross_validation(
    model,
    initial='90 days',
    period='30 days',
    horizon='7 days',
    parallel="processes"
)

# Calcular performance metrics
df_perf = performance_metrics(df_cv)

print(f'\n✅ Cross-Validation realizada:')
print(f'   - Folds: {df_cv["cutoff"].nunique()}')
print(f'   - Total de predições avaliadas: {len(df_perf)}')
print(f'   - Horizonte: até 7 dias à frente')

# Calcular métricas agregadas por horizon
metrics_by_horizon = df_perf.groupby('horizon').agg({
    'mape': ['mean', 'std'],
    'rmse': ['mean', 'std']
}).round(4)

print('\n📊 Métricas de CV por Horizon:')
print(metrics_by_horizon)

# Log das métricas agregadas no MLflow (na run ativa anterior)
with mlflow.start_run(run_id=run_id):
    # Métricas gerais de CV
    cv_mape_mean = df_perf['mape'].mean()
    cv_mape_std = df_perf['mape'].std()
    cv_rmse_mean = df_perf['rmse'].mean()
    cv_rmse_std = df_perf['rmse'].std()
    
    mlflow.log_metrics({
        'cv_mape_mean': float(cv_mape_mean),
        'cv_mape_std': float(cv_mape_std),
        'cv_rmse_mean': float(cv_rmse_mean),
        'cv_rmse_std': float(cv_rmse_std),
        'cv_folds': df_cv["cutoff"].nunique()
    })
    
    # Métricas por horizon
    for horizon_days in sorted(df_perf['horizon'].dt.days.unique()):
        horizon_data = df_perf[df_perf['horizon'].dt.days == horizon_days]
        mlflow.log_metrics({
            f'cv_mape_d{horizon_days}': float(horizon_data['mape'].mean()),
            f'cv_rmse_d{horizon_days}': float(horizon_data['rmse'].mean())
        })
    
    # Gráfico de performance por horizon
    fig_cv, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # MAPE por horizon
    mape_by_h = df_perf.groupby('horizon')['mape'].agg(['mean', 'std'])
    mape_by_h.index = mape_by_h.index.days
    ax1.errorbar(mape_by_h.index, mape_by_h['mean'], yerr=mape_by_h['std'], 
                 fmt='o-', linewidth=2, markersize=8, capsize=5, label='CV MAPE')
    ax1.axhline(y=mape_by_h['mean'].mean(), color='red', linestyle='--', 
                label=f'Mean: {mape_by_h["mean"].mean():.2%}')
    ax1.set_xlabel('Forecast Horizon (days)')
    ax1.set_ylabel('MAPE')
    ax1.set_title('Cross-Validation: MAPE by Horizon')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # RMSE por horizon
    rmse_by_h = df_perf.groupby('horizon')['rmse'].agg(['mean', 'std'])
    rmse_by_h.index = rmse_by_h.index.days
    ax2.errorbar(rmse_by_h.index, rmse_by_h['mean'], yerr=rmse_by_h['std'], 
                 fmt='s-', linewidth=2, markersize=8, capsize=5, color='green', label='CV RMSE')
    ax2.axhline(y=rmse_by_h['mean'].mean(), color='red', linestyle='--', 
                label=f'Mean: {rmse_by_h["mean"].mean():.0f}')
    ax2.set_xlabel('Forecast Horizon (days)')
    ax2.set_ylabel('RMSE (incidentes)')
    ax2.set_title('Cross-Validation: RMSE by Horizon')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    
    # Salvar figura de CV
    temp_cv_fig = os.path.join(tempfile.gettempdir(), 'prophet_cv_performance.png')
    fig_cv.savefig(temp_cv_fig, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(temp_cv_fig, 'evaluation')
    plt.close(fig_cv)
    
    print(f'\n✅ Cross-Validation metrics logadas no MLflow')
    print(f'   - CV MAPE (média): {cv_mape_mean:.2%} ± {cv_mape_std:.2%}')
    print(f'   - CV RMSE (média): {cv_rmse_mean:.0f} ± {cv_rmse_std:.0f} incidentes')
    print(f'   - Gráfico de performance por horizon logado')

Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.


🔄 Executando Cross-Validation do Prophet (este processo leva alguns minutos)...

✅ Cross-Validation realizada:
   - Folds: 8
   - Total de predições avaliadas: 7
   - Horizonte: até 7 dias à frente

📊 Métricas de CV por Horizon:
           mape          rmse    
           mean std      mean std
horizon                          
1 days   0.4113 NaN   94.6851 NaN
2 days   0.4495 NaN  101.8299 NaN
3 days   0.3999 NaN  126.9355 NaN
4 days   0.3748 NaN  123.8088 NaN
5 days   0.4344 NaN  114.7746 NaN
6 days   0.3700 NaN  255.3627 NaN
7 days   0.6934 NaN  341.1785 NaN

✅ Cross-Validation metrics logadas no MLflow
   - CV MAPE (média): 44.76% ± 11.22%
   - CV RMSE (média): 166 ± 95 incidentes
   - Gráfico de performance por horizon logado
🏃 View run prophet_v1_aggregated_daily at: https://mlflow.looplyai.com.br/#/experiments/10/runs/be5464a8de7943a4a05eeddcc5f3d853
🧪 View experiment at: https://mlflow.looplyai.com.br/#/experiments/10


In [ ]:
# Salvar resultados em data/ml/ (com caminho hardcoded)
import os
from pathlib import Path

# Usar caminho ABSOLUTO (hardcoded)
base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast')
base_path.mkdir(parents=True, exist_ok=True)

print(f'Salvando em: {base_path}')
print(f'Pasta existe: {base_path.exists()}')
print(f'Dados disponíveis:')
print(f'  - eval shape: {eval.shape}')
print(f'  - forecast_future shape: {forecast_future.shape}')

# Salvar forecast D+1 a D+7
try:
    forecast_d = forecast_future[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_d.columns = ['data', 'forecast', 'lower_ci', 'upper_ci']
    forecast_d['data'] = forecast_d['data'].dt.strftime('%Y-%m-%d')
    forecast_d['forecast'] = forecast_d['forecast'].round(0).astype(int)
    filepath = base_path / 'forecast_d1_d7.csv'
    forecast_d.to_csv(str(filepath), index=False)
    print(f'✅ Saved D+1 to D+7: {filepath}')
except Exception as e:
    print(f'❌ Erro ao salvar forecast: {e}')

# Salvar métricas de avaliação no teste
try:
    eval_results = eval.copy()
    eval_results['erro_pct'] = ((eval_results['y'] - eval_results['yhat']).abs() / (eval_results['y'] + 1) * 100).fillna(0)
    filepath = base_path / 'prophet_test_metrics.csv'
    eval_results.to_csv(str(filepath), index=False)
    print(f'✅ Saved test metrics: {filepath}')
except Exception as e:
    print(f'❌ Erro ao salvar métricas: {e}')

# Resumo
try:
    summary = pd.DataFrame({
        'métrica': ['MAPE', 'MAE', 'Dados de treino', 'Dados de teste'],
        'valor': [f'{mape:.2%}', f'{mae:.0f}', len(train), len(test)]
    })
    filepath = base_path / 'prophet_summary.csv'
    summary.to_csv(str(filepath), index=False)
    print(f'✅ Saved summary: {filepath}')
except Exception as e:
    print(f'❌ Erro ao salvar resumo: {e}')

# Verificar se os arquivos foram criados
print('\n📋 Arquivos criados:')
for file in base_path.glob('*.csv'):
    print(f'  ✅ {file.name}')

Salvando em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast
Pasta existe: True
Dados disponíveis:
  - eval shape: (30, 2)
  - forecast_future shape: (7, 28)
✅ Saved D+1 to D+7: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast\forecast_d1_d7.csv
✅ Saved test metrics: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast\prophet_test_metrics.csv
✅ Saved summary: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast\prophet_summary.csv

📋 Arquivos criados:
  ✅ forecast_d1_d7.csv
  ✅ prophet_summary.csv
  ✅ prophet_test_metrics.csv
